In [4]:
import requests
import pandas as pd
import os

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

def fetch_egfr_bioactivity(limit=1000):
    """Fetch EGFR IC50 bioactivity data from ChEMBL REST API."""
    url = f"{BASE_URL}/activity.json"
    params = {
        "target_chembl_id": "CHEMBL203",
        "standard_type":    "IC50",
        "limit":            limit,
        "offset":           0,
    }
    response = requests.get(url, params=params)
    data = response.json()
    records = data["activities"]
    df = pd.DataFrame(records)
    return df

df_raw = fetch_egfr_bioactivity(limit=2000)
print(f"Fetched {len(df_raw)} bioactivity records for EGFR")

# Create folders automatically
os.makedirs("data/raw", exist_ok=True)

# Save CSV file
df_raw.to_csv("data/raw/egfr_ic50_raw.csv", index=False)

print("CSV file saved successfully!")

Fetched 1000 bioactivity records for EGFR
CSV file saved successfully!


In [5]:
import pandas as pd
import numpy as np

# Load raw ChEMBL data
df = pd.read_csv("data/raw/egfr_ic50_raw.csv")

# Keep only essential columns
cols = ['molecule_chembl_id', 'canonical_smiles', 'standard_value',
        'standard_units', 'standard_type', 'pchembl_value']
df = df[cols].copy()

# Remove rows without SMILES or activity
df = df.dropna(subset=['canonical_smiles', 'standard_value'])

# Convert IC50 to pIC50 (–log10 scale, standard in QSAR)
# Filter out extreme outliers first
df = df[df['standard_value'] > 0]
df['pIC50'] = -np.log10(df['standard_value'].astype(float) * 1e-9)  # nM → M → –log10

# Clip to realistic drug range
df = df[(df['pIC50'] >= 3) & (df['pIC50'] <= 12)]

# Binary activity label: pIC50 >= 6 = active (IC50 ≤ 1 µM)
df['active'] = (df['pIC50'] >= 6).astype(int)

print(f"Clean dataset: {len(df)} compounds")
print(f"Active: {df['active'].sum()} | Inactive: {(df['active']==0).sum()}")
print(df[['pIC50', 'active']].describe())

Clean dataset: 965 compounds
Active: 514 | Inactive: 451
            pIC50      active
count  965.000000  965.000000
mean     6.168366    0.532642
std      1.651658    0.499192
min      3.045757    0.000000
25%      4.720000    0.000000
50%      6.124939    1.000000
75%      7.318759    1.000000
max     11.221849    1.000000
